# SIH26006 — Phase 8 & Phase 8G Master Google Colab Notebook

This self-contained notebook executes:
1. **Phase 8 Deep Learning (GRU & LSTM) Fixed Benchmark Experiments**
2. **Phase 8A–8F Optimization Pipeline** (Feature Selection, Target Transforms, Residual Hybridization, NNLS Ensembling)
3. **Phase 8G Walk-Forward Robustness Validation** across 5 Historical Market Regimes ($2021\text{--}2025$)

### Guarantees:
- **Strict Zero-Lookahead**: All scalers, feature selectors, model estimators, and NNLS ensemble weights fit strictly on historical data prior to each evaluation window.
- **Explicit Dtype Cast**: Converts feature matrices to `np.float64` to eliminate pandas warnings.
- **Unscaled Evaluation**: Metric evaluation (MAE, RMSE, sMAPE, R², Directional Accuracy) in original rate units ($/day).

In [ ]:
# ================================================================================
# CELL 1: GPU DETECTION & REPOSITORY SETUP
# ================================================================================
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Available: {gpus[0].name}")
else:
    print("Running on CPU (GPU recommended for faster training).")

if not os.path.exists('outputs/modeling_dataset.csv'):
    if os.path.exists('FICOS-Platform/outputs/modeling_dataset.csv'):
        %cd FICOS-Platform
    else:
        !git clone https://github.com/SSOHEB/FICOS-Platform.git
        %cd FICOS-Platform

os.makedirs('outputs', exist_ok=True)
print(f"Current Working Directory: {os.getcwd()}")
print(f"Dataset exists: {os.path.exists('outputs/modeling_dataset.csv')}")


In [ ]:
# ================================================================================
# CELL 2: GRU & LSTM BENCHMARK EXPERIMENTS
# ================================================================================
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv('outputs/modeling_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

feature_cols = [c for c in df.columns if not c.startswith('target_') and c not in ['date']]
df[feature_cols] = df[feature_cols].astype(np.float64)

n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

def build_gru_model(input_shape):
    return Sequential([
        Input(shape=input_shape),
        GRU(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])

def build_lstm_model(input_shape):
    return Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])

def calc_metrics(y_true_raw, y_pred_raw, y_base_raw):
    mae = np.mean(np.abs(y_true_raw - y_pred_raw))
    rmse = np.sqrt(np.mean((y_true_raw - y_pred_raw)**2))
    smape = np.mean(200 * np.abs(y_pred_raw - y_true_raw) / (np.abs(y_true_raw) + np.abs(y_pred_raw) + 1e-8))
    ss_tot = np.sum((y_true_raw - np.mean(y_true_raw))**2)
    ss_res = np.sum((y_true_raw - y_pred_raw)**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    
    actual_dir = np.sign(y_true_raw - y_base_raw)
    pred_dir = np.sign(y_pred_raw - y_base_raw)
    dir_acc = np.mean(actual_dir == pred_dir)
    return mae, rmse, smape, r2, dir_acc

TARGETS = ['kdci', 'cape', 'panamax', 'supramax', 'handy']
HORIZONS = [1, 7, 14, 30]
LOOKBACK = 30

deep_results = []
print("Starting GRU & LSTM Benchmarking Across 5 Targets x 4 Horizons...")

for tgt in TARGETS:
    for h in HORIZONS:
        target_col = f"target_{tgt}_{h}d"
        prev_col = tgt
        
        scaled_df = df.copy()
        scaler_X = StandardScaler()
        scaled_df.iloc[:n_train, scaled_df.columns.get_indexer(feature_cols)] = scaler_X.fit_transform(scaled_df.iloc[:n_train][feature_cols])
        scaled_df.iloc[n_train:, scaled_df.columns.get_indexer(feature_cols)] = scaler_X.transform(scaled_df.iloc[n_train:][feature_cols])
        
        scaler_y = StandardScaler()
        train_target = scaled_df.iloc[:n_train][target_col].values.reshape(-1, 1)
        valid_idx = ~np.isnan(train_target.flatten())
        scaler_y.fit(train_target[valid_idx])
        scaled_df[target_col + '_scaled'] = scaler_y.transform(scaled_df[target_col].values.reshape(-1, 1)).flatten()
        
        X_seq, y_seq, base_seq, idx_seq = [], [], [], []
        feat_values = scaled_df[feature_cols].values
        target_values = scaled_df[target_col + '_scaled'].values
        raw_target_values = df[target_col].values
        raw_prev_values = df[prev_col].values
        
        for i in range(LOOKBACK - 1, len(df)):
            if not np.isnan(target_values[i]):
                X_seq.append(feat_values[i - LOOKBACK + 1:i + 1])
                y_seq.append(target_values[i])
                base_seq.append(raw_prev_values[i])
                idx_seq.append(i)
                
        X_seq = np.array(X_seq)
        y_seq = np.array(y_seq)
        base_seq = np.array(base_seq)
        idx_seq = np.array(idx_seq)
        
        train_mask_seq = idx_seq < n_train
        val_mask_seq = (idx_seq >= n_train) & (idx_seq < n_train + n_val)
        test_mask_seq = idx_seq >= n_train + n_val
        
        X_train_s, y_train_s = X_seq[train_mask_seq], y_seq[train_mask_seq]
        X_val_s, y_val_s = X_seq[val_mask_seq], y_seq[val_mask_seq]
        X_test_s, y_test_s = X_seq[test_mask_seq], y_seq[test_mask_seq]
        
        raw_y_test = raw_target_values[idx_seq[test_mask_seq]]
        base_y_test = base_seq[test_mask_seq]
        
        gru = build_gru_model((LOOKBACK, len(feature_cols)))
        gru.compile(optimizer='adam', loss='mse')
        gru.fit(X_train_s, y_train_s, validation_data=(X_val_s, y_val_s), epochs=30, batch_size=32, verbose=0, callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
        
        pred_gru_scaled = gru.predict(X_test_s, verbose=0).flatten()
        pred_gru_raw = scaler_y.inverse_transform(pred_gru_scaled.reshape(-1, 1)).flatten()
        gru_mae, gru_rmse, gru_smape, gru_r2, gru_dir = calc_metrics(raw_y_test, pred_gru_raw, base_y_test)
        
        deep_results.append({
            'freight_class': tgt, 'horizon': f'{h}d', 'model': 'GRU',
            'test_MAE': round(gru_mae, 2), 'test_RMSE': round(gru_rmse, 2),
            'test_sMAPE': round(gru_smape, 2), 'test_R2': round(gru_r2, 4),
            'test_Directional_Accuracy': f'{gru_dir * 100:.1f}%', 'sample_count': len(raw_y_test)
        })
        
        lstm = build_lstm_model((LOOKBACK, len(feature_cols)))
        lstm.compile(optimizer='adam', loss='mse')
        lstm.fit(X_train_s, y_train_s, validation_data=(X_val_s, y_val_s), epochs=30, batch_size=32, verbose=0, callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
        
        pred_lstm_scaled = lstm.predict(X_test_s, verbose=0).flatten()
        pred_lstm_raw = scaler_y.inverse_transform(pred_lstm_scaled.reshape(-1, 1)).flatten()
        lstm_mae, lstm_rmse, lstm_smape, lstm_r2, lstm_dir = calc_metrics(raw_y_test, pred_lstm_raw, base_y_test)
        
        deep_results.append({
            'freight_class': tgt, 'horizon': f'{h}d', 'model': 'LSTM',
            'test_MAE': round(lstm_mae, 2), 'test_RMSE': round(lstm_rmse, 2),
            'test_sMAPE': round(lstm_smape, 2), 'test_R2': round(lstm_r2, 4),
            'test_Directional_Accuracy': f'{lstm_dir * 100:.1f}%', 'sample_count': len(raw_y_test)
        })
        
        print(f"  {tgt.upper()} {h}d: GRU sMAPE={gru_smape:.2f}% (R2={gru_r2:.4f}, Dir={gru_dir*100:.1f}%) | LSTM sMAPE={lstm_smape:.2f}% (R2={lstm_r2:.4f}, Dir={lstm_dir*100:.1f}%)")

df_deep = pd.DataFrame(deep_results)
df_deep.to_csv('outputs/phase8_gru_lstm_report.csv', index=False)
print("Saved outputs/phase8_gru_lstm_report.csv")


In [ ]:
# ================================================================================
# CELL 3: RUN PHASE 8A-8F OPTIMIZATION PIPELINE
# ================================================================================
!python src/optimization_pipeline.py


In [ ]:
# ================================================================================
# CELL 4: RUN PHASE 8G WALK-FORWARD ROBUSTNESS VALIDATION
# ================================================================================
!python src/walkforward_validation.py


In [ ]:
# ================================================================================
# CELL 5: DISPLAY WALK-FORWARD ROBUSTNESS SUMMARY MATRIX
# ================================================================================
import pandas as pd
if os.path.exists('outputs/phase8_walkforward/walkforward_summary.csv'):
    wf_sum = pd.read_csv('outputs/phase8_walkforward/walkforward_summary.csv')
    print("================================================================================")
    print("PHASE 8G WALK-FORWARD ROBUSTNESS SUMMARY (5 HISTORICAL MARKET REGIMES)")
    print("================================================================================")
    ens_wf = wf_sum[wf_sum['model'] == 'Validation_Weighted_Ensemble']
    display(ens_wf[['target', 'horizon', 'mean_sMAPE', 'std_sMAPE', 'best_sMAPE', 'worst_sMAPE', 'mean_R2', 'mean_vs_persistence_improvement', 'stability_class']])
else:
    print("outputs/phase8_walkforward/walkforward_summary.csv not found.")
